# OSM: vías principales

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/externos/osm_vias_principales.geojson`. Una fila representa un tramo OSM de vía principal, no una vía completa ni una ruta de viaje.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/externos/osm_vias_principales.geojson'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '08_osm_vias_principales', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Longitudes, tipologías y condiciones de circulación

In [ ]:
metros=g.to_crs(CRS_METROS).length
vias=df.assign(longitud_m=metros)
tipos=vias.groupby("highway").agg(tramos=("osm_id","size"),longitud_km=("longitud_m",lambda s:s.sum()/1000)).reset_index()
display(tipos)
display(metros.describe(percentiles=[.01,.5,.95,.99]).to_frame("longitud_m"))
display(faltantes(df,["name","oneway","lanes","maxspeed","surface","bridge","tunnel","access","maxheight","maxweight"]))
for c in ["oneway","bridge","tunnel","access"]:
 if c in df:display(conteos(df[c],c).head(12))
grafico_barras(tipos[["highway","tramos"]],"Tramos por jerarquía vial", "08_tipos")
resumen.update({"tipos":tipos.to_dict("records"),"longitud_total_km":float(metros.sum()/1000),
                "tramos_longitud_cero":int(metros.eq(0).sum()),"osm_id_duplicados":int(df.osm_id.duplicated().sum()),
                "longitud_descriptivos_m":metros.describe(percentiles=[.01,.5,.95,.99]).to_dict()})

## Reglas decididas para plata

- **Incorporar para proximidad y posibles interferencias con vías principales.** Mantener `osm_id`, jerarquía y geometría de tramo.
- Calcular longitud y distancias en EPSG:9377; conservar atributos de sentido, puentes, túneles y restricciones sin rellenarlos como si fueran conocidos.
- Esta selección omite calles locales y no constituye una red de rutas. No reportar tiempos de ambulancia ni caminos accesibles con ella.
- La proximidad de un edificio es una señal para verificar bloqueo, no prueba de que ocurrió ni de que pueda ocurrir exactamente a una altura de distancia.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('08_osm_vias_principales', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")